# StyleFeatureEditor Inference

## Loading repository and enviroment

Before start do not forget to choose gpu runtime (Runtime -> Change runtime type -> T4 GPU)

In [1]:
from envs import set_notebook_env_ids, change_exp_dir
CODE_DIR="/home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/SOTA_encoders_StyleGAN/StyleFeatureEditor-CS/"
set_notebook_env_ids(
    cuda_version="12.5"
)
change_exp_dir(CODE_DIR)

CUDA environment variables set for CUDA 12.5
Set TORCH_CUDA_ARCH_LIST to 8.0
Current working directory is:
 /home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/SOTA_encoders_StyleGAN/StyleFeatureEditor-CS


In [ ]:
import torchvision.transforms as transforms
import torch
import os

from utils.common_utils import tensor2im, get_keys, visualize_batch_grid
from utils.class_registry import ClassRegistry
from datasets.transforms import transforms_registry
from datasets.datasets import ImageDataset
from datasets.loaders import InfiniteLoader
from configs.data_paths import DATASETS
import torch.nn.functional as F

code_dir = "/home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/SOTA_encoders_StyleGAN/StyleFeatureEditor-CS"

Paths = {
        "code_dir": code_dir,
        "sfe_pSp_cs": code_dir + "/experiments/fse_cs_editor_train/pSp_encoder/two_directions_002/iteration_400000.pt",
        "sfe_e4e_cs": code_dir + "/experiments/fse_cs_editor_train/e4e_encoder/two_directions_doubleE_001/iteration_380000.pt", 
        "fs_cs_w_dir": "../FeatureStyleEncoder/results/12layers_ls256_lr0.001_wf/checkpoints/iteration_resume_30000.pt", 
        "images_x_path" : "/home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/special_images/background", 
        "images_y_path" : "/home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/special_images/glasses", 
        "device": "cuda",
        
}

device = Paths["device"]

import os
import yaml
encoder_name = "pSp"  # or "e4e"
model_ckpt_path = Paths[f"sfe_{encoder_name}_cs"]

# Replace the checkpoint filename with config.yaml
config_yaml_path = os.path.join(os.path.dirname(model_ckpt_path), "config.yaml")

print(config_yaml_path)

from runners.simple_runner import SimpleRunner


# runner = SimpleRunner(
#     editor_ckpt_pth="pretrained_models/sfe_editor_light.pt",
#     #editor_ckpt_pth=model_ckpt_path,
#     simple_config_pth=config_yaml_path,
#     w_space_encoder = encoder_name  
# )

runner = SimpleRunner(
    editor_ckpt_pth="pretrained_models/sfe_editor_light.pt",
    w_space_encoder = "e4e"  
)

sfe_model = runner.inference_runner
device = sfe_model.device



/home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/SOTA_encoders_StyleGAN/StyleFeatureEditor-CS/experiments/fse_cs_editor_train/pSp_encoder/two_directions_002/config.yaml
Device: cuda:0
Using e4e as w encoder, loading e4e-cs model ....
loading cs model from path pretrained_models/100000_iter_e4e_cs.pt
Loading discriminator from pretrained_models/stylegan2-ffhq-config-f.pkl
✅ discriminator re-instantiated via state_dict
Loading from checkpoint: pretrained_models/sfe_editor_light.pt
Can not find Discriminator weights in checkpoint, leave default weights.
Loading Decoder from pretrained_models/stylegan2-ffhq-config-f.pt
Loading E4E from pretrained_models/e4e_ffhq_encode.pt
✅ PID: 2872772 (check this in `nvidia-smi`)


NameError: name 'gc' is not defined

In [4]:
import os
import gc
import torch
import pynvml

# Initialize pynvml for full GPU stats
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)  # GPU:0

# Print PID for matching in nvidia-smi
print(f"✅ PID: {os.getpid()} (check this in `nvidia-smi`)")

# Force garbage collection and clear CUDA cache
gc.collect()
torch.cuda.empty_cache()

# Memory reporter
def print_memory(prefix=""):
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    used = mem_info.used / 1024**3
    total = mem_info.total / 1024**3
    print(f"[{prefix}]")
    print(f"  - PyTorch Allocated : {allocated:.2f} GB")
    print(f"  - PyTorch Reserved  : {reserved:.2f} GB")
    print(f"  - Total GPU Used    : {used:.2f} GB / {total:.2f} GB\n")

# After loading model
print_memory("After loading model")


✅ PID: 2872772 (check this in `nvidia-smi`)
[After loading model]
  - PyTorch Allocated : 3.25 GB
  - PyTorch Reserved  : 3.45 GB
  - Total GPU Used    : 17.80 GB / 40.00 GB



## Model Setup

In [5]:
from fs_models.load_models import load_FS_model, load_cs_model, load_opts_from_checkpoint
model_path = Paths["fs_cs_w_dir"]
print(f'loading opts from training checkpoint {model_path}')
previous_train_ckpt = torch.load(model_path, map_location='cpu', weights_only=True)
opts = load_opts_from_checkpoint(previous_train_ckpt)	


trainer = load_FS_model(device)

cs_mlp_net = load_cs_model(opts, model_path=model_path, device=device)


loading opts from training checkpoint ../FeatureStyleEncoder/results/12layers_ls256_lr0.001_wf/checkpoints/iteration_resume_30000.pt
{'batch_size': 4,
 'cs_checkpoint_path': './results/12layers_ls256_lr0.001_wf/checkpoints/iteration_80000.pt',
 'dataset_type': 'ffhq_glasses',
 'device': 'cuda:0',
 'exp_dir': 'results/12layers_ls256_lr0.001_wf_resume',
 'exp_scheme': 'train_cs',
 'feature_dim': 512,
 'id_lambda': 0.1,
 'image_interval': 2000,
 'img_lambda': 1.0,
 'img_loss_resize': True,
 'init_type': 'half',
 'lat_recon_lambda': 1.0,
 'learning_rate': 0.001,
 'log_interval': 2000,
 'lpips_lambda': 0.8,
 'lpips_net_type': 'alex',
 'max_steps': 500000,
 'n_layers_mlp': 12,
 'output_size': 1024,
 'pix_lambda': 1.0,
 'recon_img_target': 'both_w_real',
 'save_interval': 10000,
 'sbg_lambda': 1.0,
 'special_idx': 1,
 'style_dim': 18,
 'sx_optm_type': 'hard',
 'val_interval': 2000,
 'workers': 4}
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/home/ids/yuhe/anaconda3/envs/styleGANenv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ids/yuhe/anaconda3/envs/styleGANenv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/ids/yuhe/anaconda3/envs/styleGANenv/lib/python3.11/site-packages/lpips/weights/v0.1/alex.pth
Load pretrained model
loading csmlp weights from ../FeatureStyleEncoder/results/12layers_ls256_lr0.001_wf/checkpoints/iteration_resume_30000.pt


In [6]:
print_memory("After loading model")


[After loading model]
  - PyTorch Allocated : 4.45 GB
  - PyTorch Reserved  : 4.67 GB
  - Total GPU Used    : 19.02 GB / 40.00 GB



In [12]:

def configure_datasets(config, test_special_images):
    print("Loading dataset")
    transforms = transforms_registry[config.data.transform]().get_transforms()
    ds_name    = config.data.dataset
    cfg        = DATASETS.get(ds_name)
    if cfg is None:
        raise ValueError(f"Unknown dataset: {ds_name!r}")

    if test_special_images:
        images_x_path = Paths["images_x_path"]
        images_y_path = Paths["images_y_path"]          
    else:
        images_x_path = cfg["val_bg"]
        images_y_path = cfg["val_t"]

    # now cfg looks like {"train_bg": "...", "train_t": "...", ...}

    test_dataset_bg = ImageDataset(images_x_path, transforms["test"])
    test_dataset_t  = ImageDataset(images_y_path, transforms["test"])

    test_loader_bg = InfiniteLoader(
        test_dataset_bg,
        batch_size=config.model.batch_size,
        shuffle=False,
        num_workers=config.model.workers,
        is_infinite=False
    )
    test_loader_t = InfiniteLoader(
        test_dataset_t,
        batch_size=config.model.batch_size,
        shuffle=False,
        num_workers=config.model.workers,
        is_infinite=False
    )

    return test_loader_bg, test_loader_t

test_bg_dataloader, test_t_dataloader =configure_datasets(sfe_model.config, test_special_images=True)
X = next(iter(test_bg_dataloader)).to(device)  # or next(X) if InfiniteLoader is a generator
Y = next(iter(test_t_dataloader)).to(device)


Loading dataset


In [8]:
print_memory("After loading model")

[After loading model]
  - PyTorch Allocated : 4.54 GB
  - PyTorch Reserved  : 4.77 GB
  - Total GPU Used    : 19.12 GB / 40.00 GB



In [ ]:

@torch.inference_mode()
def calculate_delta(sfe_model, x, x_cond):
    # sample X_E as training input and X'_E as training target
    if sfe_model.config.model.w_space_encoder == "pSp":
        _, w_x = sfe_model.method.pSp_encoder(x, return_latents=True)
        _, w_x_cond = sfe_model.method.pSp_encoder(x_cond, return_latents=True)
        c_x, s_x = sfe_model.method.pSp_cs_model(w_x)
        c_y, s_y = sfe_model.method.pSp_cs_model(w_x_cond)

    elif sfe_model.config.model.w_space_encoder == "e4e":
        w_x = sfe_model.method.e4e_encoder(x) + sfe_model.method.latent_avg
        w_x_cond = sfe_model.method.e4e_encoder(x_cond) + sfe_model.method.latent_avg
        c_x, s_x = sfe_model.method.e4e_cs_model(w_x)
        c_y, s_y = sfe_model.method.e4e_cs_model(w_x_cond)

    elif sfe_model.config.model.w_space_encoder == "fs":
        w_x, _ = trainer.encode(x)
        w_x_cond, _ = trainer.encode(x_cond)
        c_x, s_x = cs_mlp_net(w_x)
        c_y, s_y = cs_mlp_net(w_x_cond)        
    else:
        raise ValueError("Unsupported w_space_encoder") 

    encoder_recon, fx = sfe_model.method.decoder(
        [c_x + s_x],
        input_is_latent=True,
        randomize_noise=False,
        return_latents=False,
        return_features=True
    )

    encoder_swap, fy = sfe_model.method.decoder(
        [c_x + s_y], 
        is_stylespace=False,
        input_is_latent=True,
        randomize_noise=False,
        return_features=True
    )

    delta = fx[9] - fy[9]
    
    return delta, encoder_recon, encoder_swap


@torch.inference_mode()
def swap_by_delta(sfe_model, x, x_cond, n_iter=1e5):
    x_resh = F.interpolate(x, size=(256, 256), mode="bilinear", align_corners=False)
    x_cond_resh = F.interpolate(x_cond, size=(256, 256), mode="bilinear", align_corners=False)

    delta, encoder_recon, encoder_swap = calculate_delta(sfe_model, x_resh, x_cond_resh)

    w_recon, predicted_feat = sfe_model.method.inverter.fs_backbone(x_resh)
    w_recon = w_recon + sfe_model.method.latent_avg
            
    _, w_feats = sfe_model.method.decoder(
        [w_recon],
        input_is_latent=True,
        return_features=True,
        is_stylespace=False,
        randomize_noise=False,
        early_stop=64
    )

    w_feat = w_feats[9]  # bs x 512 x 64 x 64 
    
    fused_feat = sfe_model.method.inverter.fuser(torch.cat([predicted_feat, w_feat], dim=1))
    if delta is None:
        delta = torch.zeros_like(fused_feat)  # inversion case
    
    edited_feat = sfe_model.method.encoder(torch.cat([fused_feat, delta], dim=1))
    feats = [None] * 9 + [edited_feat] + [None] * (17 - 9)

    images, _ = sfe_model.method.decoder(
        [w_recon],
        input_is_latent=True,
        return_features=True,
        new_features=feats,
        feature_scale=min(1.0, 0.0001 * n_iter),
        is_stylespace=False,
        randomize_noise=False
    )

    return images, encoder_recon, encoder_swap


In [18]:
from tqdm import tqdm
sfe_model.config.model.w_space_encoder = "fs"  # or "e4e", "fs"
with torch.no_grad():
    sfe_model.method.eval()

    rec_X = sfe_model.method(X)
    rec_Y = sfe_model.method(Y)

    swap_X2Y, pSp_rec_X, pSp_swap_X2Y = swap_by_delta(sfe_model, X, Y)
    swap_Y2X, pSp_rec_Y, pSp_swap_Y2X = swap_by_delta(sfe_model, Y, X)

    for idx in range(X.shape[0]):
        row1 = torch.stack([X[idx], pSp_rec_X[idx], pSp_swap_X2Y[idx], rec_X[idx], swap_X2Y[idx]], dim=0)
        row2 = torch.stack([Y[idx], pSp_rec_Y[idx], pSp_swap_Y2X[idx], rec_Y[idx], swap_Y2X[idx]], dim=0)
        # Assuming row1 and row2 are [5, C, H, W]
        columns = [torch.stack([row1[i], row2[i]], dim=0) for i in range(len(row1))]
        %matplotlib inline
        visualize_batch_grid(
            image_batches=columns,  # list of [2, C, H, W]
            titles=["Input", "pSp-cs Recon", "pSp-cs Swap", "Refined Recon", "Refined Swap"],
            save_path=None
        )    

RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 64 but got size 16 for tensor number 1 in the list.